In [37]:
!pip install -q torch==2.4.1 triton==3.0.0

In [38]:
import torch
import triton
import triton.language as tl

## Preprocessing

In [47]:
def prune_nm(
    mat: torch.Tensor,
    N: int,
    M: int,
    *,
    dim: int = 1,
    mode: str = "vw",
    block_rows: int | None = None,
    block_cols: int | None = None,
):
    """
    Prune to balanced N:M sparsity (EW / VW / BW), matching nmSPARSE figure.

    mode="ew": element-wise balanced across the flattened matrix; enforce N
                nonzeros per contiguous M elements in row-major order.
    mode="vw": vector-wise along K per row; enforce N per contiguous M elements
                along dim=1 (common 2:4 pattern for GEMM).
    mode="bw": block-wise; enforce N nonzeros *inside each block* of shape
                (block_rows x block_cols), flattened.

    Returns (pruned, mask, block_counts per block).
    """
    if mat.dim() != 2:
        raise ValueError("prune_nm expects a 2D tensor (M, K)")
    if dim != 1:
        raise ValueError("Only dim=1 (K dimension) is supported")
    if not (0 <= N <= M):
        raise ValueError("Require 0 <= N <= M for N:M sparsity")

    M_rows, K_cols = mat.shape

    if mode == "ew":
        flat = mat.reshape(-1)
        pad = (M - flat.numel() % M) % M
        if pad:
            flat = torch.nn.functional.pad(flat, (0, pad))
        flat_blocks = flat.view(-1, M)
        scores = flat_blocks.abs()
        topk_idx = scores.topk(k=N, dim=1).indices
        mask_blocks = torch.zeros_like(flat_blocks, dtype=torch.bool)
        mask_blocks.scatter_(1, topk_idx, True)
        pruned_flat = flat_blocks * mask_blocks
        pruned_flat = pruned_flat.reshape(-1)[: M_rows * K_cols]
        mask_flat = mask_blocks.reshape(-1)[: M_rows * K_cols]
        pruned = pruned_flat.view(M_rows, K_cols)
        mask = mask_flat.view(M_rows, K_cols)
        block_counts = mask_blocks.sum(dim=1)
        return pruned, mask, block_counts

    if mode == "vw":
        pad_c = (M - K_cols % M) % M
        mat_p = torch.nn.functional.pad(mat, (0, pad_c)) if pad_c else mat
        Kp = mat_p.shape[1]
        num_blocks = Kp // M

        blocks = mat_p.view(M_rows, num_blocks, M)
        scores = blocks.abs()
        topk_idx = scores.topk(k=N, dim=2).indices
        mask_blocks = torch.zeros_like(blocks, dtype=torch.bool)
        mask_blocks.scatter_(2, topk_idx, True)

        pruned_blocks = blocks * mask_blocks
        pruned_p = pruned_blocks.view(M_rows, Kp)
        mask_p = mask_blocks.view(M_rows, Kp)

        pruned = pruned_p[:, :K_cols] if pad_c else pruned_p
        mask = mask_p[:, :K_cols] if pad_c else mask_p
        block_counts = mask_blocks.view(-1, M).sum(dim=1)
        return pruned, mask, block_counts

    if mode == "bw":
        if block_rows is None or block_cols is None:
            raise ValueError("block_rows and block_cols required for bw mode")
        pad_r = (block_rows - M_rows % block_rows) % block_rows
        pad_c = (block_cols - K_cols % block_cols) % block_cols
        mat_p = torch.nn.functional.pad(mat, (0, pad_c, 0, pad_r)) if (pad_r or pad_c) else mat

        Mp, Kp = mat_p.shape
        num_br = Mp // block_rows
        num_bc = Kp // block_cols

        blocks = mat_p.view(num_br, block_rows, num_bc, block_cols).permute(0, 2, 1, 3)
        flat = blocks.reshape(num_br, num_bc, block_rows * block_cols)
        scores = flat.abs()
        topk_idx = scores.topk(k=N, dim=2).indices

        mask_flat = torch.zeros_like(flat, dtype=torch.bool)
        mask_flat.scatter_(2, topk_idx, True)
        mask_blocks = mask_flat.view(num_br, num_bc, block_rows, block_cols)
        mask_blocks = mask_blocks.permute(0, 2, 1, 3)

        pruned_blocks = blocks * mask_blocks
        pruned_p = pruned_blocks.permute(0, 2, 1, 3).reshape(num_br * block_rows, num_bc * block_cols)
        mask_p = mask_blocks.permute(0, 2, 1, 3).reshape(num_br * block_rows, num_bc * block_cols)

        pruned = pruned_p[:M_rows, :K_cols] if (pad_r or pad_c) else pruned_p
        mask = mask_p[:M_rows, :K_cols] if (pad_r or pad_c) else mask_p
        block_counts = mask_flat.reshape(-1, block_rows * block_cols).sum(dim=1)
        return pruned, mask, block_counts

    raise ValueError("mode must be 'ew', 'vw', or 'bw'")


In [49]:
def pack_blocks(
    mat: torch.Tensor,
    *,
    mode: str = "vw",
    block_size: int | None = None,
    block_rows: int | None = None,
    block_cols: int | None = None,
):
    """
    Pack pruned matrices for EW / VW / BW layouts (matches prune_nm modes).

    mode="ew": flatten row-major, group by block_size along the flat axis.
    mode="vw": per-row blocks of block_size along K.
    mode="bw": 2D blocks of (block_rows x block_cols), flattened per block.

    Returns vals, idx_in_block, block_ptr, block_coords (row_start, k_start).
    """
    if mat.dim() != 2:
        raise ValueError("pack_blocks expects a 2D tensor (M, K)")

    M_rows, K_cols = mat.shape

    if mode == "ew":
        if block_size is None:
            raise ValueError("block_size required for ew mode")
        flat = mat.reshape(-1)
        pad = (block_size - flat.numel() % block_size) % block_size
        if pad:
            flat = torch.nn.functional.pad(flat, (0, pad))
        flat_blocks = flat.view(-1, block_size)
        mask = flat_blocks != 0
        counts = mask.sum(dim=1)
        block_ptr = torch.cat([
            torch.zeros(1, device=mat.device, dtype=torch.long),
            counts.cumsum(dim=0)
        ], dim=0)
        nz = mask.nonzero(as_tuple=False)
        block_idx = nz[:, 0]
        idx_in_block = nz[:, 1]
        vals = flat_blocks[block_idx, idx_in_block]

        # map block start back to (row, col)
        block_starts = torch.arange(flat_blocks.shape[0], device=mat.device) * block_size
        rows = (block_starts // K_cols).clamp(max=M_rows - 1)
        cols = block_starts % K_cols
        block_coords = torch.stack([rows, cols], dim=1)
        return vals, idx_in_block, block_ptr, block_coords

    if mode == "vw":
        if block_size is None:
            raise ValueError("block_size required for vw mode")
        pad_c = (block_size - K_cols % block_size) % block_size
        mat_p = torch.nn.functional.pad(mat, (0, pad_c)) if pad_c else mat
        Kp = mat_p.shape[1]
        num_blocks = Kp // block_size

        blocks_flat = mat_p.view(M_rows, num_blocks, block_size).reshape(-1, block_size)
        mask = blocks_flat != 0
        counts = mask.sum(dim=1)
        block_ptr = torch.cat([
            torch.zeros(1, device=mat.device, dtype=torch.long),
            counts.cumsum(dim=0)
        ], dim=0)

        nz = mask.nonzero(as_tuple=False)
        block_idx = nz[:, 0]
        idx_in_block = nz[:, 1]
        vals = blocks_flat[block_idx, idx_in_block]

        block_rows_idx = torch.arange(M_rows, device=mat.device).unsqueeze(1).expand(M_rows, num_blocks).reshape(-1)
        block_kstart = (torch.arange(num_blocks, device=mat.device) * block_size).unsqueeze(0).expand(M_rows, num_blocks).reshape(-1)
        block_coords = torch.stack([block_rows_idx, block_kstart], dim=1)
        return vals, idx_in_block, block_ptr, block_coords

    if mode == "bw":
        if block_rows is None or block_cols is None:
            raise ValueError("block_rows and block_cols required for bw mode")
        pad_r = (block_rows - M_rows % block_rows) % block_rows
        pad_c = (block_cols - K_cols % block_cols) % block_cols
        mat_p = torch.nn.functional.pad(mat, (0, pad_c, 0, pad_r)) if (pad_r or pad_c) else mat

        Mp, Kp = mat_p.shape
        num_br = Mp // block_rows
        num_bc = Kp // block_cols

        blocks = mat_p.view(num_br, block_rows, num_bc, block_cols).permute(0, 2, 1, 3)
        flat = blocks.reshape(num_br * num_bc, block_rows * block_cols)
        mask = flat != 0
        counts = mask.sum(dim=1)
        block_ptr = torch.cat([
            torch.zeros(1, device=mat.device, dtype=torch.long),
            counts.cumsum(dim=0)
        ], dim=0)

        nz = mask.nonzero(as_tuple=False)
        block_idx = nz[:, 0]
        idx_in_block = nz[:, 1]
        vals = flat[block_idx, idx_in_block]

        br_grid, bc_grid = torch.meshgrid(
            torch.arange(num_br, device=mat.device),
            torch.arange(num_bc, device=mat.device),
            indexing="ij",
        )
        block_rows_start = (br_grid * block_rows).reshape(-1)
        block_cols_start = (bc_grid * block_cols).reshape(-1)
        block_coords = torch.stack([block_rows_start, block_cols_start], dim=1)
        return vals, idx_in_block, block_ptr, block_coords

    raise ValueError("mode must be 'ew', 'vw', or 'bw'")


def unpack_to_2d_tensor(
    vals: torch.Tensor,
    idx_in_block: torch.Tensor,
    block_ptr: torch.Tensor,
    block_coords: torch.Tensor,
    *,
    M_rows: int,
    K_cols: int,
    mode: str = "vw",
    block_size: int | None = None,
    block_rows: int | None = None,
    block_cols: int | None = None,
):
    """
    Convert packed sparse representation back to 2D tensor format.

    Takes the outputs of pack_blocks and reconstructs the original 2D matrix.

    Args:
        vals: Non-zero values (1D tensor)
        idx_in_block: Indices within each block (1D tensor)
        block_ptr: Pointers to start of each block in vals (1D tensor)
        block_coords: Coordinates of each block (N_blocks, 2) with [row_start, col_start]
        M_rows: Number of rows in the output tensor
        K_cols: Number of columns in the output tensor
        mode: Packing mode ('ew', 'vw', or 'bw')
        block_size: Block size for 'ew' and 'vw' modes
        block_rows: Block rows for 'bw' mode
        block_cols: Block columns for 'bw' mode

    Returns:
        2D tensor of shape (M_rows, K_cols) with values placed back in their positions
    """
    if vals.dim() != 1 or idx_in_block.dim() != 1 or block_ptr.dim() != 1 or block_coords.dim() != 2:
        raise ValueError("All inputs should be 1D tensors except block_coords which should be 2D")

    device = vals.device
    result = torch.zeros(M_rows, K_cols, dtype=vals.dtype, device=device)

    num_blocks = block_coords.shape[0]

    if mode == "ew":
        if block_size is None:
            raise ValueError("block_size required for ew mode")

        for block_idx in range(num_blocks):
            start = block_ptr[block_idx]
            end = block_ptr[block_idx + 1]
            block_vals = vals[start:end]
            block_indices = idx_in_block[start:end]

            row_start = block_coords[block_idx, 0]
            col_start = block_coords[block_idx, 1]

            # Convert block indices to absolute positions
            for i, idx in enumerate(block_indices):
                flat_pos = col_start + idx.item()
                row = row_start + (flat_pos // K_cols)
                col = flat_pos % K_cols
                if row < M_rows and col < K_cols:
                    result[row, col] = block_vals[i]

    elif mode == "vw":
        if block_size is None:
            raise ValueError("block_size required for vw mode")

        for block_idx in range(num_blocks):
            start = block_ptr[block_idx]
            end = block_ptr[block_idx + 1]
            block_vals = vals[start:end]
            block_indices = idx_in_block[start:end]

            row = block_coords[block_idx, 0]
            col_start = block_coords[block_idx, 1]

            # Place values directly using row and col_start + indices
            for i, idx in enumerate(block_indices):
                col = col_start + idx.item()
                if col < K_cols:
                    result[row, col] = block_vals[i]

    elif mode == "bw":
        if block_rows is None or block_cols is None:
            raise ValueError("block_rows and block_cols required for bw mode")

        for block_idx in range(num_blocks):
            start = block_ptr[block_idx]
            end = block_ptr[block_idx + 1]
            block_vals = vals[start:end]
            block_indices = idx_in_block[start:end]

            row_start = block_coords[block_idx, 0]
            col_start = block_coords[block_idx, 1]

            # Convert flattened block indices back to 2D block coordinates
            for i, flat_idx in enumerate(block_indices):
                block_row = flat_idx.item() // block_cols
                block_col = flat_idx.item() % block_cols
                row = row_start + block_row
                col = col_start + block_col
                if row < M_rows and col < K_cols:
                    result[row, col] = block_vals[i]

    else:
        raise ValueError("mode must be 'ew', 'vw', or 'bw'")

    return result


## Examples

### EW / VW / BW demos

In [50]:
# EW demo: N=2 per 4 contiguous elements (flattened, row-major)
N, M_block = 2, 4
A_ew = torch.tensor([
    [1.0, 4.0, -0.5, 0.2,   3.0, -6.0, 2.5, 0.1],
    [0.3, -2.2, 5.0, 1.1,   -7.0, 0.4, 0.2, 3.3],
    [2.0, 0.5, -4.0, 6.0,   -1.0, -3.0, 0.6, 0.7],
], dtype=torch.float32)

pruned_ew, mask_ew, counts_ew = prune_nm(A_ew, N, M_block, mode="ew")
print("EW original:\n", A_ew)
print("EW mask (1=keep):\n", mask_ew.int())
print("EW pruned:\n", pruned_ew)
print("EW block_counts:", counts_ew)
vals_ew, idx_ew, ptr_ew, coords_ew = pack_blocks(pruned_ew, mode="ew", block_size=M_block)
print("EW vals:", vals_ew)
print("EW idx:", idx_ew)
print("EW ptr:", ptr_ew)
print("EW coords (row, col-start):", coords_ew)

EW original:
 tensor([[ 1.0000,  4.0000, -0.5000,  0.2000,  3.0000, -6.0000,  2.5000,  0.1000],
        [ 0.3000, -2.2000,  5.0000,  1.1000, -7.0000,  0.4000,  0.2000,  3.3000],
        [ 2.0000,  0.5000, -4.0000,  6.0000, -1.0000, -3.0000,  0.6000,  0.7000]])
EW mask (1=keep):
 tensor([[1, 1, 0, 0, 1, 1, 0, 0],
        [0, 1, 1, 0, 1, 0, 0, 1],
        [0, 0, 1, 1, 1, 1, 0, 0]], dtype=torch.int32)
EW pruned:
 tensor([[ 1.0000,  4.0000, -0.0000,  0.0000,  3.0000, -6.0000,  0.0000,  0.0000],
        [ 0.0000, -2.2000,  5.0000,  0.0000, -7.0000,  0.0000,  0.0000,  3.3000],
        [ 0.0000,  0.0000, -4.0000,  6.0000, -1.0000, -3.0000,  0.0000,  0.0000]])
EW block_counts: tensor([2, 2, 2, 2, 2, 2])
EW vals: tensor([ 1.0000,  4.0000,  3.0000, -6.0000, -2.2000,  5.0000, -7.0000,  3.3000,
        -4.0000,  6.0000, -1.0000, -3.0000])
EW idx: tensor([0, 1, 0, 1, 1, 2, 0, 3, 2, 3, 0, 1])
EW ptr: tensor([ 0,  2,  4,  6,  8, 10, 12])
EW coords (row, col-start): tensor([[0, 0],
        [0, 4],
   

In [42]:
# VW demo: N=2 per 4 along K (per row)
N, M_block = 2, 4
A_vw = torch.tensor([
    [1.0, -5.0, 3.0, 0.2,   -0.5, 4.5, 2.0, -3.0],
    [0.1, 2.5, -4.0, 6.0,   1.5, -0.2, -2.5, 5.5],
    [3.2, -1.0, 0.5, -2.2,  4.0, -6.0, 1.0, 0.8],
], dtype=torch.float32)

pruned_vw, mask_vw, counts_vw = prune_nm(A_vw, N, M_block, mode="vw")
print("VW original:\n", A_vw)
print("VW mask (1=keep):\n", mask_vw.int())
print("VW pruned:\n", pruned_vw)
print("VW block_counts:", counts_vw)
vals_vw, idx_vw, ptr_vw, coords_vw = pack_blocks(pruned_vw, mode="vw", block_size=M_block)
print("VW vals:", vals_vw)
print("VW idx:", idx_vw)
print("VW ptr:", ptr_vw)
print("VW coords (row, k-start):", coords_vw)

VW original:
 tensor([[ 1.0000, -5.0000,  3.0000,  0.2000, -0.5000,  4.5000,  2.0000, -3.0000],
        [ 0.1000,  2.5000, -4.0000,  6.0000,  1.5000, -0.2000, -2.5000,  5.5000],
        [ 3.2000, -1.0000,  0.5000, -2.2000,  4.0000, -6.0000,  1.0000,  0.8000]])
VW mask (1=keep):
 tensor([[0, 1, 1, 0, 0, 1, 0, 1],
        [0, 0, 1, 1, 0, 0, 1, 1],
        [1, 0, 0, 1, 1, 1, 0, 0]], dtype=torch.int32)
VW pruned:
 tensor([[ 0.0000, -5.0000,  3.0000,  0.0000, -0.0000,  4.5000,  0.0000, -3.0000],
        [ 0.0000,  0.0000, -4.0000,  6.0000,  0.0000, -0.0000, -2.5000,  5.5000],
        [ 3.2000, -0.0000,  0.0000, -2.2000,  4.0000, -6.0000,  0.0000,  0.0000]])
VW block_counts: tensor([2, 2, 2, 2, 2, 2])
VW vals: tensor([-5.0000,  3.0000,  4.5000, -3.0000, -4.0000,  6.0000, -2.5000,  5.5000,
         3.2000, -2.2000,  4.0000, -6.0000])
VW idx: tensor([1, 2, 1, 3, 2, 3, 2, 3, 0, 3, 0, 1])
VW ptr: tensor([ 0,  2,  4,  6,  8, 10, 12])
VW coords (row, k-start): tensor([[0, 0],
        [0, 4],
     

In [51]:
# BW demo: N=4 nonzeros per 2x4 block (element-wise inside block)
N_block, BR, BC = 4, 2, 4
A_bw = torch.tensor([
    [1.0, -2.0, 3.0, -4.0,   5.0, -6.0, 7.0, -8.0],
    [0.5, 6.5, -1.5, 2.5,    -3.5, 4.5, -5.5, 6.5],
    [9.0, -1.0, 0.2, -0.3,   2.2, -2.3, 1.1, -1.2],
    [-4.4, 3.3, -2.2, 1.1,   0.9, -0.8, 0.7, -0.6],
], dtype=torch.float32)

pruned_bw, mask_bw, counts_bw = prune_nm(A_bw, N_block, BR * BC, mode="bw", block_rows=BR, block_cols=BC)
print("BW original:\n", A_bw)
print("BW mask (1=keep):\n", mask_bw.int())
print("BW pruned:\n", pruned_bw)
print("BW block_counts:", counts_bw)
vals_bw, idx_bw, ptr_bw, coords_bw = pack_blocks(pruned_bw, mode="bw", block_rows=BR, block_cols=BC)
print("BW vals:", vals_bw)
print("BW idx:", idx_bw)
print("BW ptr:", ptr_bw)
print("BW coords (row_start, k_start):", coords_bw)

BW original:
 tensor([[ 1.0000, -2.0000,  3.0000, -4.0000,  5.0000, -6.0000,  7.0000, -8.0000],
        [ 0.5000,  6.5000, -1.5000,  2.5000, -3.5000,  4.5000, -5.5000,  6.5000],
        [ 9.0000, -1.0000,  0.2000, -0.3000,  2.2000, -2.3000,  1.1000, -1.2000],
        [-4.4000,  3.3000, -2.2000,  1.1000,  0.9000, -0.8000,  0.7000, -0.6000]])
BW mask (1=keep):
 tensor([[0, 0, 1, 1, 0, 1, 0, 1],
        [0, 1, 1, 1, 0, 0, 0, 1],
        [1, 0, 0, 0, 1, 1, 1, 0],
        [1, 1, 1, 1, 0, 0, 0, 0]], dtype=torch.int32)
BW pruned:
 tensor([[ 0.0000, -0.0000,  3.0000, -4.0000,  0.0000, -6.0000,  0.0000, -8.0000],
        [ 0.0000,  6.5000, -1.5000,  2.5000, -0.0000,  0.0000, -0.0000,  6.5000],
        [ 9.0000, -0.0000,  0.0000, -0.0000,  2.2000, -2.3000,  1.1000, -0.0000],
        [-4.4000,  3.3000, -2.2000,  1.1000,  0.0000, -0.0000,  0.0000, -0.0000]])
BW block_counts: tensor([4, 4, 4, 4])
BW vals: tensor([ 3.0000, -4.0000,  6.5000, -1.5000,  2.5000, -6.0000, -8.0000,  6.5000,
         9.000

In [40]:
# Data Reshaping (paper-style condensed tensors)
# Paper Fig.3 stores condensed tensors in "N-rows" layout:
#   Data  : (N, M_rows * Number_of_Blocks)
#   Index : (N, M_rows * Number_of_Blocks)
# Each column is a block (row-major over blocks). Row i stores the i-th kept
# element within each block.

def to_paper_condensed(
    vals: torch.Tensor,
    idx_in_block: torch.Tensor,
    *,
    M_rows: int,
    Number_of_Blocks: int,
    N: int,
    idx_format: str = "in_block",  # "in_block" (paper) or "absolute"
    block_size: int | None = None,  # required if idx_format="absolute"
):
    """Convert flat packed outputs to the paper's condensed format (Fig.3).

    Inputs are the *flat* outputs of pack_blocks (assuming balanced N entries per block):
    - vals: 1D, length = M_rows * Number_of_Blocks * N
    - idx_in_block: 1D, same length, offsets inside each block (0..block_size-1)

    Returns:
    - data:  (N, M_rows * Number_of_Blocks)
    - index: (N, M_rows * Number_of_Blocks)
    """

    if vals.dim() != 1 or idx_in_block.dim() != 1:
        raise ValueError(
            f"Expected 1D vals/idx. Got vals.dim={vals.dim()}, idx.dim={idx_in_block.dim()}"
        )

    M_rows = int(M_rows)
    Number_of_Blocks = int(Number_of_Blocks)
    N = int(N)

    blocks_total = M_rows * Number_of_Blocks
    expected = blocks_total * N

    if vals.numel() != expected or idx_in_block.numel() != expected:
        raise ValueError(
            f"Expected vals/idx length {expected} (= M_rows*Number_of_Blocks*N); "
            f"got vals={vals.numel()}, idx={idx_in_block.numel()}."
        )

    # Group by block: (blocks_total, N)
    data_bn = vals.reshape(blocks_total, N)
    idx_bn = idx_in_block.reshape(blocks_total, N)

    if idx_format == "absolute":
        if block_size is None:
            raise ValueError("block_size is required when idx_format='absolute'")
        # Block id -> (row, block_in_row)
        block_in_row = (torch.arange(blocks_total, device=idx_bn.device) % Number_of_Blocks).to(idx_bn.dtype)
        offsets = (block_in_row * int(block_size)).view(blocks_total, 1)
        idx_bn = idx_bn + offsets
    elif idx_format != "in_block":
        raise ValueError("idx_format must be 'in_block' or 'absolute'")

    # Paper layout: (N, blocks_total)
    return data_bn.T.contiguous(), idx_bn.T.contiguous()


def to_row_major_condensed(data_paper: torch.Tensor, index_paper: torch.Tensor, *, M_rows: int, Number_of_Blocks: int):
    """Optional helper: convert paper layout back to (M_rows, Number_of_Blocks*N)."""
    M_rows = int(M_rows)
    Number_of_Blocks = int(Number_of_Blocks)
    N = int(data_paper.shape[0])
    blocks_total = M_rows * Number_of_Blocks
    if data_paper.shape != (N, blocks_total) or index_paper.shape != (N, blocks_total):
        raise ValueError("Unexpected paper tensor shapes")
    data_bn = data_paper.T.contiguous().reshape(M_rows, Number_of_Blocks * N)
    idx_bn = index_paper.T.contiguous().reshape(M_rows, Number_of_Blocks * N)
    return data_bn, idx_bn


def pack_to_2d_tensor(
    vals: torch.Tensor,
    idx_in_block: torch.Tensor,
    block_ptr: torch.Tensor,
    block_coords: torch.Tensor,
    *,
    mode: str = "vw",
    block_size: int | None = None,
    block_rows: int | None = None,
    block_cols: int | None = None,
):
    """
    Convert packed sparse representation to 2D tensor format for data and indices.

    This function takes the outputs of pack_blocks and organizes them into 2D tensors
    where each column represents a block, and each row within a column represents
    the stored elements/indices for that block.

    Args:
        vals: Non-zero values (1D tensor)
        idx_in_block: Indices within each block (1D tensor)
        block_ptr: Pointers to start of each block in vals (1D tensor)
        block_coords: Coordinates of each block (N_blocks, 2) with [row_start, col_start]
        mode: Packing mode ('ew', 'vw', or 'bw')
        block_size: Block size for 'ew' and 'vw' modes
        block_rows: Block rows for 'bw' mode
        block_cols: Block columns for 'bw' mode

    Returns:
        data_2d: 2D tensor of shape (max_block_size, num_blocks) containing the values
        idx_2d: 2D tensor of shape (max_block_size, num_blocks) containing the indices
        block_coords: Same as input block_coords
    """
    if vals.dim() != 1 or idx_in_block.dim() != 1 or block_ptr.dim() != 1 or block_coords.dim() != 2:
        raise ValueError("All inputs should be 1D tensors except block_coords which should be 2D")

    device = vals.device
    num_blocks = block_coords.shape[0]

    # Determine max block size (N)
    if mode == "ew":
        if block_size is None:
            raise ValueError("block_size required for ew mode")
        max_block_size = block_size
    elif mode == "vw":
        if block_size is None:
            raise ValueError("block_size required for vw mode")
        max_block_size = block_size
    elif mode == "bw":
        if block_rows is None or block_cols is None:
            raise ValueError("block_rows and block_cols required for bw mode")
        max_block_size = block_rows * block_cols
    else:
        raise ValueError("mode must be 'ew', 'vw', or 'bw'")

    # Initialize 2D tensors
    data_2d = torch.zeros(max_block_size, num_blocks, dtype=vals.dtype, device=device)
    idx_2d = torch.zeros(max_block_size, num_blocks, dtype=idx_in_block.dtype, device=device)

    # Fill the 2D tensors
    for block_idx in range(num_blocks):
        start = block_ptr[block_idx]
        end = block_ptr[block_idx + 1]
        block_vals = vals[start:end]
        block_indices = idx_in_block[start:end]
        actual_size = end - start

        # Store values and indices in the corresponding column
        data_2d[:actual_size, block_idx] = block_vals
        idx_2d[:actual_size, block_idx] = block_indices

    return data_2d, idx_2d, block_coords


In [41]:
# Test: paper-format condensation using the VW demo outputs
# VW: block_size=4, N=2, Number_of_Blocks=2 (K=8)

# Paper format (in-block indices like Fig.3):
data_p, idx_p = to_paper_condensed(
    vals_vw,
    idx_vw,
    M_rows=A_vw.shape[0],
    Number_of_Blocks=2,
    N=2,
    idx_format="in_block",
)

print("data_p shape:", tuple(data_p.shape))  # (N, M_rows*Number_of_Blocks)
print("idx_p  shape:", tuple(idx_p.shape))
print("idx_p (paper, in-block) first 4 blocks:\n", idx_p[:, :4])

# If you also want absolute column indices (NOT what Fig.3 prints):
# data_abs, idx_abs = to_paper_condensed(
#     vals_vw,
#     idx_vw,
#     M_rows=A_vw.shape[0],
#     Number_of_Blocks=2,
#     N=2,
#     idx_format="absolute",
#     block_size=4,
# )
# print("idx_abs first 4 blocks:\n", idx_abs[:, :4])


# Test the unpack_to_2d_tensor function
print("Testing unpack_to_2d_tensor function:")

# Test VW mode
reconstructed_vw = unpack_to_2d_tensor(
    vals_vw, idx_vw, ptr_vw, coords_vw,
    M_rows=A_vw.shape[0], K_cols=A_vw.shape[1],
    mode="vw", block_size=4
)
print("VW mode - Original pruned:")
print(pruned_vw)
print("VW mode - Reconstructed:")
print(reconstructed_vw)
print("VW mode - Are they equal?", torch.allclose(pruned_vw, reconstructed_vw))

# Test EW mode
reconstructed_ew = unpack_to_2d_tensor(
    vals_ew, idx_ew, ptr_ew, coords_ew,
    M_rows=A_ew.shape[0], K_cols=A_ew.shape[1],
    mode="ew", block_size=4
)
print("\nEW mode - Original pruned:")
print(pruned_ew)
print("EW mode - Reconstructed:")
print(reconstructed_ew)
print("EW mode - Are they equal?", torch.allclose(pruned_ew, reconstructed_ew))

# Test BW mode
reconstructed_bw = unpack_to_2d_tensor(
    vals_bw, idx_bw, ptr_bw, coords_bw,
    M_rows=A_bw.shape[0], K_cols=A_bw.shape[1],
    mode="bw", block_rows=2, block_cols=4
)
print("\nBW mode - Original pruned:")
print(pruned_bw)
print("BW mode - Reconstructed:")
print(reconstructed_bw)
print("BW mode - Are they equal?", torch.allclose(pruned_bw, reconstructed_bw))

print("\nTesting pack_to_2d_tensor function:")

# Test VW mode
data_2d_vw, idx_2d_vw, coords_vw_out = pack_to_2d_tensor(
    vals_vw, idx_vw, ptr_vw, coords_vw,
    mode="vw", block_size=4
)
print("VW mode - shapes:", data_2d_vw.shape, idx_2d_vw.shape)
print("VW data:\n", data_2d_vw)
print("VW idx:\n", idx_2d_vw)

# Test EW mode
data_2d_ew, idx_2d_ew, coords_ew_out = pack_to_2d_tensor(
    vals_ew, idx_ew, ptr_ew, coords_ew,
    mode="ew", block_size=4
)
print("\nEW mode - shapes:", data_2d_ew.shape, idx_2d_ew.shape)
print("EW data:\n", data_2d_ew)
print("EW idx:\n", idx_2d_ew)

# Test BW mode
data_2d_bw, idx_2d_bw, coords_bw_out = pack_to_2d_tensor(
    vals_bw, idx_bw, ptr_bw, coords_bw,
    mode="bw", block_rows=2, block_cols=4
)
print("\nBW mode - shapes:", data_2d_bw.shape, idx_2d_bw.shape)
print("BW data:\n", data_2d_bw)
print("BW idx:\n", idx_2d_bw)


data_p shape: (2, 6)
idx_p  shape: (2, 6)
idx_p (paper, in-block) first 4 blocks:
 tensor([[1, 1, 2, 2],
        [2, 3, 3, 3]])
Testing unpack_to_2d_tensor function:
VW mode - Original pruned:
tensor([[ 0.0000, -5.0000,  3.0000,  0.0000, -0.0000,  4.5000,  0.0000, -3.0000],
        [ 0.0000,  0.0000, -4.0000,  6.0000,  0.0000, -0.0000, -2.5000,  5.5000],
        [ 3.2000, -0.0000,  0.0000, -2.2000,  4.0000, -6.0000,  0.0000,  0.0000]])
VW mode - Reconstructed:
tensor([[ 0.0000, -5.0000,  3.0000,  0.0000,  0.0000,  4.5000,  0.0000, -3.0000],
        [ 0.0000,  0.0000, -4.0000,  6.0000,  0.0000,  0.0000, -2.5000,  5.5000],
        [ 3.2000,  0.0000,  0.0000, -2.2000,  4.0000, -6.0000,  0.0000,  0.0000]])
VW mode - Are they equal? True

EW mode - Original pruned:
tensor([[ 1.0000,  4.0000, -0.0000,  0.0000,  3.0000, -6.0000,  0.0000,  0.0000],
        [ 0.0000, -2.2000,  5.0000,  0.0000, -7.0000,  0.0000,  0.0000,  3.3000],
        [ 0.0000,  0.0000, -4.0000,  6.0000, -1.0000, -3.0000,  0